In [1]:
!pip install transformers sentencepiece tqdm pandas numpy scikit-learn matplotlib

In [2]:
import torch
print(f"GPU        : {torch.cuda.get_device_name(0)}")
print(f"Capability : {torch.cuda.get_device_capability(0)}")

import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve,
                             precision_recall_curve, average_precision_score)
from tqdm.notebook import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

GPU        : Tesla T4
Capability : (7, 5)
Device : cuda


In [3]:
cyano = pd.read_parquet("hf://datasets/tattabio/cyano_operonic_pair/data/train-00000-of-00001.parquet")
vibrio = pd.read_parquet("hf://datasets/tattabio/vibrio_operonic_pair/data/train-00000-of-00001.parquet")
ecoli = pd.read_parquet("hf://datasets/tattabio/ecoli_operonic_pair/data/train-00000-of-00001.parquet")
tatta_bio_df = pd.concat([cyano, vibrio, ecoli],ignore_index=True)


In [4]:
def create_pairs(df: pd.DataFrame) -> pd.DataFrame:
    # Extract genome accession and positional index from entry
    df = df.copy()
    df['genome'] = df['Entry'].str.extract(r'lcl\|(.*?)_prot')
    df['idx'] = df['Entry'].str.extract(r'_(\d+)$').astype(int)
    
    # Sort by genome and position
    df = df.sort_values(['genome', 'idx']).reset_index(drop=True)
    
    pairs = []
    for genome, group in df.groupby('genome'):
        group = group.reset_index(drop=True)
        for i in range(len(group) - 1):
            # Only pair if indices are consecutive
            if group.loc[i+1, 'idx'] == group.loc[i, 'idx'] + 1:
                pairs.append({
                    'sequence_1': group.loc[i, 'Sequence'],
                    'sequence_2': group.loc[i+1, 'Sequence'],
                    'label': group.loc[i, 'Label']
                })
    
    return pd.DataFrame(pairs)
tatta_pairs = create_pairs(tatta_bio_df)
tatta_pairs.to_csv("dgeb.csv", index=False)

In [5]:
# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_CSV      = "/kaggle/input/datasets/kennethrodrigues/operon-pair-classification-v2/operon_train.csv"
VAL_CSV        = "/kaggle/input/datasets/kennethrodrigues/operon-pair-classification-v2/operon_val.csv"
TEST_CSV       = "dgeb.csv"

TRAIN_EMB_PATH = Path("train_protbert_embeddings.npz")
VAL_EMB_PATH   = Path("val_protbert_embeddings.npz")
TEST_EMB_PATH  = Path("test_protbert_embeddings.npz")

OUT_DIR        = Path("protbert_mlp_output")
OUT_DIR.mkdir(exist_ok=True)

# ── ProtBERT-BFD ──────────────────────────────────────────────────────────────
MODEL_NAME     = "Rostlab/prot_bert_bfd"
MAX_SEQ_LEN    = 512    # ProtBERT max is 1024 but 512 is safer on T4
EMB_BATCH_SIZE = 4      # 420M model — conservative on T4
CHUNK_SIZE     = 500

# ── MLP classifier ────────────────────────────────────────────────────────────
# input: concat · diff · product of (1024,) embeddings = 4 × 1024 = 4096
HIDDEN_DIMS    = [2048, 512, 128, 32]
DROPOUT        = 0.3

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE     = 128
EPOCHS         = 50
LR             = 1e-4
WEIGHT_DECAY   = 1e-2
PATIENCE       = 10

print("Config loaded.")
print(f"Model       : {MODEL_NAME}  (hidden=1024)")
print(f"Fused dim   : 4 × 1024 = 4096")
print(f"MLP dims    : 4096 → {' → '.join(map(str, HIDDEN_DIMS))} → 1")

Config loaded.
Model       : Rostlab/prot_bert_bfd  (hidden=1024)
Fused dim   : 4 × 1024 = 4096
MLP dims    : 4096 → 2048 → 512 → 128 → 32 → 1


In [6]:
df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)
df_test  = pd.read_csv(TEST_CSV)

for name, df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    print(f"{name:<6} rows: {len(df):>6}  "
          f"pos rate: {df['label'].mean():.3f}  "
          f"cols: {list(df.columns)}")

Train  rows:  37126  pos rate: 0.501  cols: ['sequence_1', 'sequence_2', 'operon_id', 'label']
Val    rows:  15912  pos rate: 0.499  cols: ['sequence_1', 'sequence_2', 'operon_id', 'label']
Test   rows:   5183  pos rate: 0.314  cols: ['sequence_1', 'sequence_2', 'label']


In [7]:
!df -h /kaggle/working
!du -sh * 2>/dev/null | sort -rh | head -15

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  3.4M   20G   1% /kaggle/working
3.2M	dgeb.csv
48K	__notebook__.ipynb
4.0K	protbert_mlp_output


In [8]:
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = (AutoModel
              .from_pretrained(MODEL_NAME)
              .to(DEVICE)
              .eval())

print(f"Hidden size : {bert_model.config.hidden_size}")   # 1024
print(f"VRAM used   : {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert_bfd
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hidden size : 1024
VRAM used   : 1.68 GB


In [9]:
def preprocess(sequences: list[str]) -> list[str]:
    """
    ProtBERT requires space-separated amino acids and
    rare/ambiguous characters replaced with X.
    """
    allowed = set("ACDEFGHIKLMNPQRSTVWY")
    result  = []
    for seq in sequences:
        seq = seq.upper()
        seq = "".join(aa if aa in allowed else "X" for aa in seq)
        result.append(" ".join(list(seq)))
    return result


@torch.no_grad()
def encode_batch(sequences: list[str]) -> np.ndarray:
    """
    Encode a batch with ProtBERT-BFD.
    Returns (B, 1024) float16 numpy array — mean pooled, CLS/EOS stripped.
    """
    processed = preprocess(sequences)

    inputs = tokenizer(
        processed,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        add_special_tokens=True,
    ).to(DEVICE)

    outputs = bert_model(**inputs)
    hidden  = outputs.last_hidden_state   # (B, L, 1024) fp32

    results = []
    for j in range(len(sequences)):
        mask     = inputs["attention_mask"][j]
        real_len = mask.sum().item()
        # strip CLS (pos 0) and SEP (last real pos)
        emb  = hidden[j, 1:real_len - 1, :]   # (seq_len, 1024)
        # mean pool
        pooled = emb.mean(dim=0)               # (1024,)
        results.append(pooled.half().cpu().numpy())   # fp16

    return np.stack(results)   # (B, 1024) float16


def encode_and_save(df: pd.DataFrame, path: Path,
                    force_rerun: bool = False) -> dict:
    """
    Encode both sequence columns in chunks and save as .npz fp16.
    Resumes from last chunk if interrupted.
    """
    if path.exists() and not force_rerun:
        print(f"Found {path} — loading.")
        data = np.load(path)
        return {"emb_a": data["emb_a"],
                "emb_b": data["emb_b"],
                "labels": data["labels"]}

    chunk_dir = path.with_suffix("")
    chunk_dir.mkdir(exist_ok=True)

    seqs_a = df["sequence_1"].tolist()
    seqs_b = df["sequence_2"].tolist()
    labels = df["label"].to_numpy().astype(np.float32)
    n      = len(df)

    for col, seqs in [("a", seqs_a), ("b", seqs_b)]:
        for start in tqdm(range(0, n, CHUNK_SIZE),
                          desc=f"  Encoding gene {col.upper()}"):
            chunk_path = chunk_dir / f"emb_{col}_{start:07d}.npy"
            if chunk_path.exists():
                continue

            end    = min(start + CHUNK_SIZE, n)
            batch  = seqs[start:end]
            chunks = []
            for i in range(0, len(batch), EMB_BATCH_SIZE):
                chunks.append(encode_batch(batch[i: i + EMB_BATCH_SIZE]))

            np.save(chunk_path, np.vstack(chunks))

    # ── merge ─────────────────────────────────────────────────────────────
    print("  Merging chunks...")
    emb_a = np.vstack([
        np.load(chunk_dir / f"emb_a_{s:07d}.npy")
        for s in range(0, n, CHUNK_SIZE)
    ])
    emb_b = np.vstack([
        np.load(chunk_dir / f"emb_b_{s:07d}.npy")
        for s in range(0, n, CHUNK_SIZE)
    ])

    np.savez_compressed(path, emb_a=emb_a, emb_b=emb_b, labels=labels)
    print(f"  Saved → {path}  ({path.stat().st_size / 1e6:.1f} MB)  "
          f"shape: {emb_a.shape}  dtype: {emb_a.dtype}")

    for f in chunk_dir.glob("*.npy"):
        f.unlink()
    chunk_dir.rmdir()

    return {"emb_a": emb_a, "emb_b": emb_b, "labels": labels}

In [10]:
print("=== Train ===")
train_data = encode_and_save(df_train, TRAIN_EMB_PATH)

print("\n=== Val ===")
val_data   = encode_and_save(df_val,   VAL_EMB_PATH)

print("\n=== Test ===")
test_data  = encode_and_save(df_test,  TEST_EMB_PATH)

for name, d in [("Train", train_data),
                ("Val",   val_data),
                ("Test",  test_data)]:
    print(f"{name:<6} emb_a: {d['emb_a'].shape}  "
          f"dtype: {d['emb_a'].dtype}  "
          f"pos rate: {d['labels'].mean():.3f}")

print("\nDisk after encoding:")
!df -h /kaggle/working
!du -sh *.npz 2>/dev/null

=== Train ===


  Encoding gene A:   0%|          | 0/75 [00:00<?, ?it/s]

  Encoding gene B:   0%|          | 0/75 [00:00<?, ?it/s]

  Merging chunks...
  Saved → train_protbert_embeddings.npz  (141.2 MB)  shape: (37126, 1024)  dtype: float16

=== Val ===


  Encoding gene A:   0%|          | 0/32 [00:00<?, ?it/s]

  Encoding gene B:   0%|          | 0/32 [00:00<?, ?it/s]

  Merging chunks...
  Saved → val_protbert_embeddings.npz  (60.5 MB)  shape: (15912, 1024)  dtype: float16

=== Test ===


  Encoding gene A:   0%|          | 0/11 [00:00<?, ?it/s]

  Encoding gene B:   0%|          | 0/11 [00:00<?, ?it/s]

  Merging chunks...
  Saved → test_protbert_embeddings.npz  (19.7 MB)  shape: (5183, 1024)  dtype: float16
Train  emb_a: (37126, 1024)  dtype: float16  pos rate: 0.501
Val    emb_a: (15912, 1024)  dtype: float16  pos rate: 0.499
Test   emb_a: (5183, 1024)  dtype: float16  pos rate: 0.314

Disk after encoding:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  215M   20G   2% /kaggle/working
19M	test_protbert_embeddings.npz
135M	train_protbert_embeddings.npz
58M	val_protbert_embeddings.npz


In [11]:
del bert_model
torch.cuda.empty_cache()
print(f"VRAM after unloading ProtBERT: {torch.cuda.memory_allocated()/1e9:.2f} GB")

VRAM after unloading ProtBERT: 0.01 GB


In [12]:
def build_features(data: dict) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Architecture B fusion: concat · diff · product → (N, 4 × 1024 = 4096)
    Casts fp16 → fp32 here, not at encode time, to keep files small.
    """
    a = torch.tensor(data["emb_a"], dtype=torch.float32)   # (N, 1024)
    b = torch.tensor(data["emb_b"], dtype=torch.float32)
    y = torch.tensor(data["labels"], dtype=torch.float32)

    X = torch.cat([a, b, a - b, a * b], dim=1)   # (N, 4096)
    return X, y


X_train, y_train = build_features(train_data)
X_val,   y_val   = build_features(val_data)
X_test,  y_test  = build_features(test_data)

input_dim = X_train.shape[1]   # 4096

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Input dim (4 × 1024) : {input_dim}")

X_train : torch.Size([37126, 4096])
X_val   : torch.Size([15912, 4096])
X_test  : torch.Size([5183, 4096])
Input dim (4 × 1024) : 4096


In [13]:
def make_loader(X, y, shuffle):
    ds = TensorDataset(X, y)
    return DataLoader(ds, batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=0)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

pos_weight = torch.tensor(
    (y_train == 0).sum() / (y_train == 1).sum(),
    dtype=torch.float32
).to(DEVICE)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")
print(f"pos_weight    : {pos_weight:.3f}")

Train batches : 291
Val   batches : 125
Test  batches : 41
pos_weight    : 0.996


/tmp/ipykernel_23/891696395.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight = torch.tensor(


In [14]:
class OperonMLP(nn.Module):
    """
    Siamese ProtBERT-BFD — Architecture B

    Both sequences encoded by ProtBERT-BFD (shared weights at encode time).
    Pair fused via concat · diff · product → (4096,).
    MLP classifies the fused vector.

    Input  : (N, 4096)  — fused pair embeddings
    Output : (N,)       — raw logit
    """
    def __init__(self, input_dim: int, hidden_dims: list, dropout: float):
        super().__init__()
        layers = []
        prev   = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


model = OperonMLP(input_dim, HIDDEN_DIMS, DROPOUT).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters : {total_params:,}")
print(f"Input dim            : {input_dim}  (4 × 1024)")

OperonMLP(
  (net): Sequential(
    (0): Linear(in_features=4096, out_features=2048, bias=True)
    (1): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=2048, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=512, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=128, out_features=32, bias=True)
    (13): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.3, inplace=False)
    (16): Linear(in_features=32, out_features=1, bias=True)
  )
)

Trainable parameters : 9,515,009
Input dim            : 4096  (4 

In [15]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

MODEL_PATH = OUT_DIR / "best_model.pt"

print(f"Criterion  : BCEWithLogitsLoss  pos_weight={pos_weight:.3f}")
print(f"Optimizer  : AdamW  lr={LR}  wd={WEIGHT_DECAY}")
print(f"Scheduler  : CosineAnnealingLR  T_max={EPOCHS}")

Criterion  : BCEWithLogitsLoss  pos_weight=0.996
Optimizer  : AdamW  lr=0.0001  wd=0.01
Scheduler  : CosineAnnealingLR  T_max=50


In [16]:
def smooth_labels(y, smoothing=0.1):
    return y * (1 - smoothing) + 0.5 * smoothing


history = {
    "train_loss": [], "val_loss": [],
    "val_auroc":  [], "val_f1":   [],
}
best_auroc    = 0.0
best_epoch    = 0
epochs_no_imp = 0

for epoch in range(1, EPOCHS + 1):

    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, smooth_labels(y_batch))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * len(y_batch)

    train_loss /= len(X_train)

    # ── Validate ──────────────────────────────────────────────────────────
    model.eval()
    val_loss  = 0.0
    val_probs = []
    val_preds = []
    val_true  = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits    = model(X_batch)
            val_loss += criterion(logits, y_batch).item() * len(y_batch)
            probs     = torch.sigmoid(logits).cpu().numpy()
            val_probs.extend(probs)
            val_preds.extend((probs >= 0.5).astype(int))
            val_true.extend(y_batch.cpu().numpy())

    val_loss  /= len(X_val)
    val_auroc  = roc_auc_score(val_true, val_probs)
    val_f1     = f1_score(val_true, val_preds)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_auroc"].append(val_auroc)
    history["val_f1"].append(val_f1)

    # ── Checkpoint + early stopping ───────────────────────────────────────
    if val_auroc > best_auroc:
        best_auroc    = val_auroc
        best_epoch    = epoch
        epochs_no_imp = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "input_dim":        input_dim,
            "hidden_dims":      HIDDEN_DIMS,
            "dropout":          DROPOUT,
            "epoch":            epoch,
            "val_auroc":        val_auroc,
            "history":          history,
        }, MODEL_PATH)
    else:
        epochs_no_imp += 1

    print(f"Epoch {epoch:>3}/{EPOCHS}  "
          f"train_loss: {train_loss:.4f}  "
          f"val_loss: {val_loss:.4f}  "
          f"val_AUROC: {val_auroc:.4f}  "
          f"val_F1: {val_f1:.4f}  "
          f"best: {best_auroc:.4f} @ {best_epoch}  "
          f"patience: {epochs_no_imp}/{PATIENCE}")

    if epochs_no_imp >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest val AUROC : {best_auroc:.4f} at epoch {best_epoch}")
print(f"Model saved    → {MODEL_PATH}")

Epoch   1/50  train_loss: 0.6445  val_loss: 0.5652  val_AUROC: 0.7827  val_F1: 0.6798  best: 0.7827 @ 1  patience: 0/10
Epoch   2/50  train_loss: 0.5855  val_loss: 0.5257  val_AUROC: 0.8137  val_F1: 0.7269  best: 0.8137 @ 2  patience: 0/10
Epoch   3/50  train_loss: 0.5545  val_loss: 0.5060  val_AUROC: 0.8321  val_F1: 0.7472  best: 0.8321 @ 3  patience: 0/10
Epoch   4/50  train_loss: 0.5290  val_loss: 0.4948  val_AUROC: 0.8412  val_F1: 0.7276  best: 0.8412 @ 4  patience: 0/10
Epoch   5/50  train_loss: 0.5071  val_loss: 0.4797  val_AUROC: 0.8491  val_F1: 0.7589  best: 0.8491 @ 5  patience: 0/10
Epoch   6/50  train_loss: 0.4896  val_loss: 0.4815  val_AUROC: 0.8506  val_F1: 0.7640  best: 0.8506 @ 6  patience: 0/10
Epoch   7/50  train_loss: 0.4728  val_loss: 0.4699  val_AUROC: 0.8560  val_F1: 0.7574  best: 0.8560 @ 7  patience: 0/10
Epoch   8/50  train_loss: 0.4575  val_loss: 0.4554  val_AUROC: 0.8655  val_F1: 0.7733  best: 0.8655 @ 8  patience: 0/10
Epoch   9/50  train_loss: 0.4402  val_lo

In [17]:
n_epochs = len(history["train_loss"])
epochs   = range(1, n_epochs + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Siamese ProtBERT-BFD — Architecture B (concat·diff·product → MLP)",
             fontsize=13, fontweight="bold")

# Loss
axes[0].plot(epochs, history["train_loss"], linewidth=1.8, label="Train loss")
axes[0].plot(epochs, history["val_loss"],   linewidth=1.8,
             linestyle="--", label="Val loss")
axes[0].axvline(best_epoch, color="tab:red", linestyle=":",
                alpha=0.6, label=f"Best epoch {best_epoch}")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

# AUROC
axes[1].plot(epochs, history["val_auroc"], color="tab:green", linewidth=1.8)
axes[1].axvline(best_epoch, color="tab:red", linestyle="--", alpha=0.6,
                label=f"Best: {best_auroc:.4f} @ epoch {best_epoch}")
axes[1].axhline(0.5, color="grey", linestyle=":", linewidth=0.8,
                label="Random baseline")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUROC")
axes[1].set_title("Validation AUROC")
axes[1].set_ylim(0.5, 1.0); axes[1].legend(); axes[1].grid(alpha=0.3)

# F1
axes[2].plot(epochs, history["val_f1"], color="tab:orange", linewidth=1.8)
axes[2].axvline(best_epoch, color="tab:red", linestyle="--", alpha=0.6,
                label=f"Best F1 @ epoch {best_epoch}")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("F1")
axes[2].set_title("Validation F1")
axes[2].set_ylim(0, 1); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
save_path = OUT_DIR / "training_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

Saved → protbert_mlp_output/training_curves.png


In [18]:
ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded checkpoint — epoch {ckpt['epoch']}  "
      f"val AUROC: {ckpt['val_auroc']:.4f}")

test_probs = []
test_preds = []
test_true  = []

with torch.no_grad():
    for X_batch, y_batch in tqdm(test_loader, desc="Testing"):
        X_batch  = X_batch.to(DEVICE)
        logits   = model(X_batch)
        probs    = torch.sigmoid(logits).cpu().numpy()
        test_probs.extend(probs)
        test_preds.extend((probs >= 0.5).astype(int))
        test_true.extend(y_batch.numpy())

test_probs = np.array(test_probs)
test_preds = np.array(test_preds)
test_true  = np.array(test_true).astype(int)

Loaded checkpoint — epoch 34  val AUROC: 0.8914


Testing:   0%|          | 0/41 [00:00<?, ?it/s]

In [19]:
fpr, tpr, roc_thresh = roc_curve(test_true, test_probs)
youden_idx  = np.argmax(tpr - fpr)
best_thresh = roc_thresh[youden_idx]
preds_opt   = (test_probs >= best_thresh).astype(int)

auroc = roc_auc_score(test_true, test_probs)
auprc = average_precision_score(test_true, test_probs)
f1    = f1_score(test_true, preds_opt)
prec  = precision_score(test_true, preds_opt)
rec   = recall_score(test_true, preds_opt)
acc   = accuracy_score(test_true, preds_opt)

print("=" * 52)
print("  Test set — Siamese ProtBERT-BFD Architecture B")
print("=" * 52)
print(f"  Threshold (Youden's J) : {best_thresh:.4f}")
print("-" * 52)
print(f"  AUROC                  : {auroc:.4f}")
print(f"  AUPRC                  : {auprc:.4f}")
print(f"  F1                     : {f1:.4f}")
print(f"  Precision              : {prec:.4f}")
print(f"  Recall                 : {rec:.4f}")
print(f"  Accuracy               : {acc:.4f}")
print("=" * 52)

  Test set — Siamese ProtBERT-BFD Architecture B
  Threshold (Youden's J) : 0.9026
----------------------------------------------------
  AUROC                  : 0.7064
  AUPRC                  : 0.5109
  F1                     : 0.5352
  Precision              : 0.5074
  Recall                 : 0.5661
  Accuracy               : 0.6913


In [20]:
cm = confusion_matrix(test_true, preds_opt)
pr_prec, pr_rec, pr_thresh = precision_recall_curve(test_true, test_probs)
closest_idx = np.argmin(np.abs(pr_thresh - best_thresh))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Test Set — Siamese ProtBERT-BFD Architecture B",
             fontsize=13, fontweight="bold")

# Confusion matrix
disp = ConfusionMatrixDisplay(cm, display_labels=["Non-operonic", "Operonic"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix")
total = cm.sum()
for (r, c), val in np.ndenumerate(cm):
    axes[0].text(c, r + 0.35, f"({100*val/total:.1f}%)",
                 ha="center", fontsize=9, color="grey")

# ROC
axes[1].plot(fpr, tpr, color="tab:blue", linewidth=2,
             label=f"AUROC = {auroc:.4f}")
axes[1].scatter(fpr[youden_idx], tpr[youden_idx],
                color="tab:red", zorder=5,
                label=f"Threshold = {best_thresh:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend(); axes[1].grid(alpha=0.3)

# Precision-Recall
axes[2].plot(pr_rec, pr_prec, color="tab:orange", linewidth=2,
             label=f"AUPRC = {auprc:.4f}")
axes[2].scatter(pr_rec[closest_idx], pr_prec[closest_idx],
                color="tab:red", zorder=5,
                label=f"Threshold = {best_thresh:.3f}")
axes[2].axhline(test_true.mean(), color="grey", linestyle="--",
                linewidth=0.8, label=f"Random ({test_true.mean():.3f})")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision–Recall Curve")
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
save_path = OUT_DIR / "test_evaluation.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

Saved → protbert_mlp_output/test_evaluation.png


In [21]:
print("Output files:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:<40} {f.stat().st_size / 1e6:.1f} MB")

print(f"\nEmbedding files (fp16):")
for p in [TRAIN_EMB_PATH, VAL_EMB_PATH, TEST_EMB_PATH]:
    if p.exists():
        print(f"  {p.name:<40} {p.stat().st_size / 1e6:.1f} MB")

print(f"\nTotal disk:")
!df -h /kaggle/working

Output files:
  best_model.pt                            38.1 MB
  test_evaluation.png                      0.1 MB
  training_curves.png                      0.1 MB

Embedding files (fp16):
  train_protbert_embeddings.npz            141.2 MB
  val_protbert_embeddings.npz              60.5 MB
  test_protbert_embeddings.npz             19.7 MB

Total disk:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  252M   20G   2% /kaggle/working
